# 04 · Entrenamiento del modelo v2

**Objetivo:** entrenar la segunda versión del detector utilizando el dataset completo de 750 imágenes revisadas manualmente, mejorando los resultados del modelo v1 entrenado con solo 152 imágenes.

**Contexto:** este es el resultado final del flujo de active learning. El dataset ha pasado de 152 a 750 imágenes etiquetadas gracias al ciclo de auto-etiquetado y revisión. Se espera una mejora general de las métricas, especialmente en la detección del balón, que era el punto más débil del modelo v1.

**Entrada:** dataset de 750 imágenes descargado de Roboflow (versión revisada).

**Salida:** modelo `best_v2.pt` y métricas comparadas con el v1.

## 1. Descarga del dataset revisado desde Roboflow

El dataset con las 750 imágenes revisadas se descarga directamente desde Roboflow en formato YOLOv8. La API key se introduce de forma segura.

**Nota:** esta descarga sobrescribe el contenido de `data/dataset/` con la versión actualizada del dataset (750 imágenes en lugar de las 152 iniciales).

In [2]:
from roboflow import Roboflow
from getpass import getpass
from pathlib import Path
import shutil
import os

REPO_ROOT = Path("../..").resolve()
DATASET_DIR = REPO_ROOT / "data" / "dataset"

WORKSPACE = "hugos-workspace-0dwnk"
PROJECT = "football-tactical-analysis"

api_key = getpass("Introduce tu API key de Roboflow: ")

In [7]:
# Eliminar dataset antiguo si existe
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
    print("Dataset antiguo eliminado")

rf = Roboflow(api_key=api_key)
project = rf.workspace(WORKSPACE).project(PROJECT)

# Descargar la versión 2 (743 imágenes revisadas)
version = project.version(2)
dataset = version.download("yolov8", location=str(DATASET_DIR))

print(f"\nDataset descargado en: {DATASET_DIR}")

Dataset antiguo eliminado
loading Roboflow workspace...
loading Roboflow project...
Exporting format yolov8 in progress : 0.0%
Version export complete for yolov8 format



Extracting Dataset Version Zip to C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\data\dataset in yolov8:: 100%|██████████| 1491/1491 [00:01<00:00, 1095.11it/s]


Dataset descargado en: C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\data\dataset


## 2. Verificación del dataset descargado

In [8]:
import torch
from ultralytics import YOLO

DATA_YAML = DATASET_DIR / "data.yaml"

train_imgs = list((DATASET_DIR / "train" / "images").glob("*.jpg"))
valid_imgs = list((DATASET_DIR / "valid" / "images").glob("*.jpg"))
test_imgs = list((DATASET_DIR / "test" / "images").glob("*.jpg")) if (DATASET_DIR / "test").exists() else []

print(f"Train: {len(train_imgs)} imágenes")
print(f"Valid: {len(valid_imgs)} imágenes")
print(f"Test:  {len(test_imgs)} imágenes")
print(f"Total: {len(train_imgs) + len(valid_imgs) + len(test_imgs)} imágenes")
print(f"\nGPU disponible: {torch.cuda.is_available()}")

Train: 520 imágenes
Valid: 149 imágenes
Test:  74 imágenes
Total: 743 imágenes

GPU disponible: True


## 3. Corrección del data.yaml

Se reescribe el `data.yaml` con rutas absolutas para garantizar que el entrenamiento funcione independientemente del directorio de ejecución.

In [9]:
test_line = f"test: {DATASET_DIR}/test/images\n" if test_imgs else ""

yaml_content = f"""train: {DATASET_DIR}/train/images
val: {DATASET_DIR}/valid/images
{test_line}nc: 3
names: ['ball', 'player', 'referee']
"""

with open(DATA_YAML, "w", encoding="utf-8") as f:
    f.write(yaml_content)

print("data.yaml actualizado:\n")
print(yaml_content)

data.yaml actualizado:

train: C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\data\dataset/train/images
val: C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\data\dataset/valid/images
test: C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\data\dataset/test/images
nc: 3
names: ['ball', 'player', 'referee']



## 4. Entrenamiento del modelo v2

Se mantienen los mismos hiperparámetros que en el v1 para que la comparación sea justa: la única variable que cambia es el tamaño del dataset (750 frente a 152 imágenes). Esto permite aislar el efecto del aumento de datos sobre el rendimiento del modelo.

In [10]:
model = YOLO("yolov8s.pt")

results = model.train(
    data=str(DATA_YAML),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project=str(REPO_ROOT / "runs" / "detect" / "models"),
    name="detector_v2",
    patience=15,
    save=True,
    plots=True,
    workers=0
)

print(f"\nEntrenamiento completado")
print(f"Resultados guardados en: {results.save_dir}")

New https://pypi.org/project/ultralytics/8.4.61 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.53  Python-3.13.13 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\data\dataset\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momen

## 5. Métricas del modelo v2

In [12]:
metrics = model.val(workers=0)

print("\nMétricas finales del modelo v2:\n")
print(f"{'Clase':<15} {'Precisión':>10} {'Recall':>10} {'mAP50':>10} {'mAP50-95':>10}")
print("-" * 60)

for i, cls in model.names.items():
    p = metrics.box.p[i]
    r = metrics.box.r[i]
    map50 = metrics.box.ap50[i]
    map5095 = metrics.box.ap[i]
    print(f"{cls:<15} {p:>10.3f} {r:>10.3f} {map50:>10.3f} {map5095:>10.3f}")

print("-" * 60)
print(f"{'Promedio':<15} {metrics.box.mp:>10.3f} {metrics.box.mr:>10.3f} {metrics.box.map50:>10.3f} {metrics.box.map:>10.3f}")

Ultralytics 8.4.53  Python-3.13.13 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
val: Fast image access  (ping: 0.20.0 ms, read: 59.47.1 MB/s, size: 56.6 KB)
val: Scanning C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\data\dataset\valid\labels.cache... 149 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 149/149 41.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 2.3it/s 4.3s.3ss
                   all        149       3439       0.86       0.78      0.807       0.48
                  ball        125        125      0.923      0.576      0.618      0.218
                player        149       3012      0.932      0.963      0.977      0.702
               referee        143        302      0.724      0.801      0.825      0.521
Speed: 1.2ms preprocess, 12.1ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\notebooks\fase2_detecci

## 6. Comparación v1 vs v2

Se comparan las métricas de ambos modelos para cuantificar la mejora obtenida gracias al aumento del dataset mediante active learning.

In [13]:
# Métricas del modelo v1 (registradas en el notebook 01)
metricas_v1 = {
    "ball":    {"mAP50": 0.551},
    "player":  {"mAP50": 0.949},
    "referee": {"mAP50": 0.962},
}

print("Comparación de mAP50 entre modelos:\n")
print(f"{'Clase':<15} {'v1 (152)':>12} {'v2 (750)':>12} {'Mejora':>10}")
print("-" * 50)

for i, cls in model.names.items():
    v1 = metricas_v1[cls]["mAP50"]
    v2 = metrics.box.ap50[i]
    mejora = v2 - v1
    print(f"{cls:<15} {v1:>12.3f} {v2:>12.3f} {mejora:>+10.3f}")

Comparación de mAP50 entre modelos:

Clase               v1 (152)     v2 (750)     Mejora
--------------------------------------------------
ball                   0.551        0.618     +0.067
player                 0.949        0.977     +0.028
referee                0.962        0.825     -0.137


## 6.1 Análisis del rendimiento por clase

La comparación entre v1 y v2 revela tres comportamientos distintos:

- **Balón (+0.067):** mejora clara gracias al refuerzo del dataset. El recall sube de 0.50 a 0.576, indicando que el modelo detecta más balones. Además, la validación del v2 contiene 125 instancias de balón frente a las 22 del v1, lo que hace la métrica mucho más fiable estadísticamente.

- **Jugador (+0.028):** mejora sobre una métrica ya alta, evaluada sobre 3012 instancias, por lo que es muy robusta.

- **Árbitro (-0.137):** descenso aparente que requiere matización. El conjunto de validación del v2 contiene 302 instancias de árbitro frente a las 43 del v1. Con tan pocos ejemplos en el v1, la métrica estaba sobreestimada; el valor del v2 es más representativo y realista del rendimiento real del modelo.

## 6.2 Identificación del problema: confusión árbitro-jugador

La matriz de confusión del modelo v2 confirma una confusión bidireccional entre las clases árbitro y jugador:

- De los 302 árbitros reales, 238 se detectan correctamente, pero 48 (un 16%) se clasifican erróneamente como jugadores.
- En sentido inverso, 58 jugadores se clasifican como árbitros.

Esta confusión se explica por la variabilidad cromática del árbitro: a diferencia de otras clases, no tiene un aspecto fijo y puede vestir colores que coinciden con la equipación de algún equipo. Dado que el modelo se apoya parcialmente en el color para distinguir clases, cuando este no es discriminante tiende a confundir ambas.

Adicionalmente, la matriz confirma que el balón sigue siendo la clase más difícil: 51 de los 125 balones del conjunto de validación no se detectan, perdiéndose contra el fondo. También se observan 277 falsos positivos de jugador correspondientes a elementos del fondo (público o publicidad).

Estos hallazgos justifican las dos líneas de mejora previstas: data augmentation de color para reducir la dependencia cromática (modelo v3), y las técnicas específicas de refuerzo del balón planificadas para fases posteriores.

In [16]:
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

results_dir = Path(results.save_dir)
cm_path = results_dir / "confusion_matrix.png"

if cm_path.exists():
    img = mpimg.imread(str(cm_path))
    plt.figure(figsize=(10, 8))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Matriz de confusión - Modelo v2")
    plt.show()
else:
    print(f"No encontrada en: {cm_path}")

<Figure size 1000x800 with 1 Axes>

## 6.3 Estrategia de mejora para el modelo v3

A partir de este análisis se decide entrenar un tercer modelo (v3) aplicando data augmentation, con dos objetivos concretos:

- **Variaciones de color (brillo, saturación, tono):** reducen la dependencia del modelo respecto al color exacto, ayudándole a distinguir árbitros y jugadores por su forma y contexto además del color. Esto ataca directamente el problema de confusión identificado.

- **Aumento del volumen efectivo de datos:** las transformaciones geométricas y de color multiplican el número de ejemplos de entrenamiento, lo que ayuda especialmente a las clases con menos instancias como el árbitro.

Esta decisión responde además a la recomendación expresa de la directora del TFG, que sugirió incorporar data augmentation dado el tamaño reducido del dataset. La comparación entre los tres modelos (v1, v2 y v3) sobre un conjunto de test común se documenta en el notebook de comparativa final.

## 7. Guardado del modelo v2

In [14]:
import shutil

source = Path(results.save_dir) / "weights" / "best.pt"
destination = REPO_ROOT / "models" / "best_v2.pt"

shutil.copy2(source, destination)

print(f"Modelo v2 guardado en: {destination}")
print(f"Tamaño: {destination.stat().st_size / (1024*1024):.2f} MB")

Modelo v2 guardado en: C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\models\best_v2.pt
Tamaño: 21.48 MB


## 8. Conclusiones

Tras este notebook:

- Se ha entrenado el modelo v2 con el dataset completo de 743 imágenes revisadas, obteniendo mejoras claras en la detección de balón y jugadores.
- El análisis por clase ha permitido identificar que el árbitro es la clase más difícil, debido a su variabilidad cromática y al desbalance de instancias, lo que provoca confusión con la clase jugador.
- A raíz de este hallazgo y de la recomendación de la directora, se planifica un modelo v3 con data augmentation para mitigar estos problemas.
- El modelo v2 queda guardado como `models/best_v2.pt`.
- La comparación rigurosa de los tres modelos se realizará sobre un conjunto de test común en el notebook de comparativa, garantizando una evaluación metodológicamente correcta.